# Abertura da RNF — Enel Distribuição São Paulo

Este notebook abre a **Renda Não Faturada (RNF)** — a energia já entregue e ainda não faturada no
fechamento do mês — por item tarifário, família e camada, com a mesma tabela de tarifas e as mesmas
famílias do P × Q do Faturamento (`SP/BW`). A visão geral está no [README](../README.md); as decisões
e pendências, no [sessions](../sessions.md).

**A fonte é o próprio relatório contábil da RNF** (`RNF CCS <mmaaaa> ... .xlsb`). O valor a abrir é
o que a contabilidade registrou; o notebook só o decompõe. A lógica é a mesma do Faturamento:

| Bloco | O que é |
|---|---|
| **1 — Calculado** | quantidade não faturada × tarifa homologada, nos 37 itens e nas 4 camadas |
| **2 — Ajustado** | o bloco 1 levado ao valor do relatório, linha a linha |

**O caminho:** 1. Parâmetros → 2. Ler o relatório → 3. Linhas do SAP (ZUBIL_146) → 4. Ajustes
manuais → 4b. Retirar PIS/COFINS → 5. Chave tarifária → 6. Posto horário (pelo BW do mês) →
7. Tarifa da vigência → 8. Bloco 1 → 9. Bloco 2 → 10. Conferências → 11. Conciliação com o
relatório → 12. Gravar.

---
## Passo 1 — Parâmetros

Todo mês muda só `REFERENCIA`; o nome do relatório segue o padrão
`RNF CCS <mmaaaa> Fator K Nova Metodologia.xlsb`. A tabela de tarifas e o P × Q do Faturamento do
mesmo mês vêm da pasta do BW: a tarifa é uma fonte única, e o P × Q fornece a divisão por posto
horário (Passo 6).

In [ ]:
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from pyxlsb import open_workbook

# ---------------------------------------------------------------- period and files
REFERENCIA = "2026-08"

BASE = Path(os.environ.get("RNF_BASE", r"C:\python_projects\claude\p_x_q\SP\RNF"))
ARQ_RELATORIO   = BASE / "src" / f"RNF CCS {REFERENCIA[5:7]}{REFERENCIA[:4]} Fator K Nova Metodologia.xlsb"
ARQ_TARIFA_LONG = BASE.parent / "BW" / "tbl" / "tarifas" / "tarifa_long.csv"   # single tariff source
ARQ_PXQ_BW      = BASE.parent / "BW" / "1-Entrega" / "pxq" / f"pxq_{REFERENCIA}.xlsx"
DIR_SAIDA       = BASE / "1-Entrega" / "rnf"
SAIDA           = DIR_SAIDA / f"rnf_{REFERENCIA}.xlsx"

# PIS/COFINS rate embedded in the report values: read from the report itself (sheet "RNF TOI SP",
# cells "Aliquota de COFINS" + "Aliquota de PIS"). Set a number here only to override it.
ALIQUOTA_PIS_COFINS = None

ALVO = pd.Timestamp(REFERENCIA + "-01")
for arq in (ARQ_RELATORIO, ARQ_TARIFA_LONG, ARQ_PXQ_BW):
    assert arq.exists(), f"arquivo nao encontrado: {arq}"
print(f"referencia {REFERENCIA} -> {SAIDA.name}")

---
## Passo 2 — Ler o relatório contábil

São usadas seis abas, todas localizadas pelo rótulo das linhas (não por posição fixa):

| Aba | O que fornece |
|---|---|
| `Relatório SAP_ZUBIL_146` | RNF do sistema, por lote, classe/subclasse e categoria tarifária: kWh, kW e R$ não faturados |
| `RNF_POR_CLASSE` | volume (kWh) de cada ajuste manual por classe |
| `RESUMO ZUBIL_146 POR GRUPO A_B` | valor (R$) de cada ajuste manual |
| `RNF TOI SP` | alíquotas de PIS e COFINS |
| `Conciliação contábil Novo` | valor contabilizado na conta 1120110910, alvo da conciliação final |
| `Comparativo Novo` | totais do mês por mercado, para comparação |

In [ ]:
def le_aba(livro, nome):
    """All rows of a sheet as dicts {column index: value}."""
    with livro.get_sheet(nome) as aba:
        return [{c.c: c.v for c in linha if c.v not in (None, "")} for linha in aba.rows()]

with open_workbook(str(ARQ_RELATORIO)) as livro:
    ABAS = {n.strip(): n for n in livro.sheets}
    zubil_bruto = le_aba(livro, ABAS["Relatório SAP_ZUBIL_146"])
    por_classe  = le_aba(livro, ABAS["RNF_POR_CLASSE"])
    resumo      = le_aba(livro, ABAS["RESUMO ZUBIL_146 POR GRUPO A_B"])
    comparativo = le_aba(livro, ABAS["Comparativo Novo"])
    toi         = le_aba(livro, ABAS["RNF TOI SP"])
    contabil    = le_aba(livro, ABAS["Conciliação contábil Novo"])

def acha_linha(linhas, rotulo, col=1):
    """First row whose label column matches (trimmed, case-insensitive)."""
    for l in linhas:
        if str(l.get(col, "")).strip().lower() == rotulo.strip().lower():
            return l
    raise KeyError(f"rotulo nao encontrado: {rotulo}")

exercicios = {str(l.get(6)).strip() for l in zubil_bruto[3:]
              if re.fullmatch(r"\d{4}/\d{2}", str(l.get(6, "")).strip())}
assert exercicios == {REFERENCIA.replace("-", "/")}, f"relatorio de outro periodo: {exercicios}"
print(f"relatorio {ARQ_RELATORIO.name}: exercicio {exercicios.pop()}, {len(zubil_bruto)} linhas no ZUBIL")

---
## Passo 3 — Linhas do SAP (ZUBIL_146)

O ZUBIL traz a RNF calculada pelo sistema para cada lote, classe/subclasse e categoria tarifária. As
linhas são somadas por classe, subclasse e categoria (o lote não muda a tarifa). Para cada uma:

- **Energia (MWh):** consumo TE no cativo; consumo TUSD no livre.
- **Demanda (kW):** demanda ativa não faturada do cativo do grupo A. A coluna de demanda do livre
  ainda não tem significado confirmado e fica fora (sessions, P2).
- **Valor (R$):** ativo TE + ativo TUSD (cativo) + ativo TUSD (livre). É o alvo do bloco 2.
- **Mercado:** segue a convenção do relatório — as colunas de TE e "TUSD TE" são cativo e a coluna
  "TUSD" é livre, qualquer que seja a categoria. Categoria livre com valor nas colunas do cativo gera
  uma linha cativa própria (sessions, P5).
- **Não tarifáveis:** reativo e bandeira vão para a aba `Nao_tarifaveis`.

As categorias tarifárias e as classes do SAP são traduzidas por tabelas explícitas: uma categoria
nova interrompe o processo, para ser mapeada aqui.

In [ ]:
# ZUBIL columns (0-based), from the header row of the sheet
COL = {"GRP": 8, "SUB": 9, "CAT": 10, "CL1": 2,
       "R_TE": 18, "R_TUSD_CAT": 20, "R_TUSD_LIV": 22, "R_REATIVO": 26,
       "KWH_TE": 34, "KWH_TUSD_LIV": 38, "KW_CAT": 48, "KW_LIV": 50, "R_BANDEIRA": 58}
cab = zubil_bruto[2]
assert str(cab.get(COL["CAT"])).strip() == "Categoria Tarifa" and \
       str(cab.get(COL["KWH_TE"])).strip() == "Consumo Ativo Não Faturado TE", "layout do ZUBIL mudou"

# category -> (subgroup, modality, market)
CATEGORIA = {
    "A2_AZU": ("A2", "Azul", "CATIVO"),      "A2_LIVREAZ": ("A2", "Azul", "LIVRE"),
    "A3A_AZU": ("A3a", "Azul", "CATIVO"),    "A3A_VDE": ("A3a", "Verde", "CATIVO"),
    "A3ALIVREAZ": ("A3a", "Azul", "LIVRE"),  "A3ALIVREVD": ("A3a", "Verde", "LIVRE"),
    "A4_AZU": ("A4", "Azul", "CATIVO"),      "A4_AZURRE": ("A4", "Azul", "CATIVO"),
    "A4_VDE": ("A4", "Verde", "CATIVO"),     "A4_VDERRE": ("A4", "Verde", "CATIVO"),
    "A4_LIVREAZ": ("A4", "Azul", "LIVRE"),   "A4_LIVREVD": ("A4", "Verde", "LIVRE"),
    "AS_AZU": ("AS", "Azul", "CATIVO"),      "AS_VDE": ("AS", "Verde", "CATIVO"),
    "AS_LIVREAZ": ("AS", "Azul", "LIVRE"),   "AS_LIVREVD": ("AS", "Verde", "LIVRE"),
    "B1_RESID": ("B1", "Convencional", "CATIVO"), "B1_RESI_TB": ("B1", "Branca", "CATIVO"),
    "B2_RUR": ("B2", "Convencional", "CATIVO"),
    "B3_AESAN": ("B3", "Convencional", "CATIVO"), "B3_OUTROS": ("B3", "Convencional", "CATIVO"),
    "B3_OUTR_TB": ("B3", "Branca", "CATIVO"),
}
# SAP main class (first three letters) -> class of the report
CLASSE = {"REP": "RESIDENCIAL", "REB": "RESIDENCIAL", "RES": "RESIDENCIAL",
          "COM": "COMERCIAL", "IND": "INDUSTRIAL", "RUR": "RURAL", "RUA": "RURAL",
          "PPE": "PODER PÚBLICO", "PPF": "PODER PÚBLICO", "PPM": "PODER PÚBLICO",
          "SPA": "SERVIÇO PÚBLICO", "SPT": "SERVIÇO PÚBLICO"}
# SAP subclass -> social benefit (PRORET 7.1 par. 18 and 23; README do BW 3.5)
BENEFICIO = {"REBXR": "BAIXA_RENDA", "REBRBPC": "BAIXA_RENDA", "REBRIND": "BAIXA_RENDA",
             "REBRQUI": "BAIXA_RENDA", "REBXDS": "DESCONTO_SOCIAL"}

def num(v):
    try:
        f = float(v)
        return f if np.isfinite(f) else 0.0
    except (TypeError, ValueError):
        return 0.0

reg = []
for l in zubil_bruto[3:]:
    grp = str(l.get(COL["GRP"], "")).strip()
    if grp not in ("ELGA", "ELGB"):            # subtotal rows have no group
        continue
    reg.append({"CL1": str(l.get(COL["CL1"], "")).strip(), "SUB_SAP": str(l.get(COL["SUB"], "")).strip(),
                "CATEGORIA": str(l.get(COL["CAT"], "")).strip(),
                **{k: num(l.get(c)) for k, c in COL.items() if k not in ("GRP", "SUB", "CAT", "CL1")}})
z = pd.DataFrame(reg)

novas = sorted(set(z["CATEGORIA"]) - set(CATEGORIA))
assert not novas, f"categoria tarifaria sem mapeamento: {novas}"
novas = sorted(set(z["CL1"]) - set(CLASSE))
assert not novas, f"classe do SAP sem mapeamento: {novas}"

VALORES = ["R_TE", "R_TUSD_CAT", "R_TUSD_LIV", "R_REATIVO", "KWH_TE", "KWH_TUSD_LIV",
           "KW_CAT", "KW_LIV", "R_BANDEIRA"]
sap = z.groupby(["CL1", "SUB_SAP", "CATEGORIA"], as_index=False)[VALORES].sum()
sap[["SUBGRUPO", "MODALIDADE", "MERCADO"]] = pd.DataFrame(sap["CATEGORIA"].map(CATEGORIA).tolist(),
                                                          index=sap.index)
sap["CLASSE_REL"] = sap["CL1"].map(CLASSE)
sap["BENEFICIO"] = np.where(sap["MODALIDADE"] == "Convencional", sap["SUB_SAP"].map(BENEFICIO), None)
# Report convention: the TE and "TUSD TE" columns are captive, the "TUSD" column is free market,
# whatever the category. A free-market category with values in the captive columns gives an extra
# captive line (energy from the TE column, demand from the "Demanda Ativa Nao Faturado" column).
cat_livre = sap["CATEGORIA"].map(lambda c: CATEGORIA[c][2]) == "LIVRE"
parte_cativa = sap[cat_livre & ((sap["R_TE"] + sap["R_TUSD_CAT"]) != 0)].copy()
parte_cativa["MERCADO"] = "CATIVO"
parte_cativa["ORIGEM"] = "SAP ZUBIL_146 (TE e TUSD TE em categoria livre)"
parte_cativa[["R_TUSD_LIV", "KWH_TUSD_LIV", "KW_LIV", "R_REATIVO", "R_BANDEIRA"]] = 0.0
sap.loc[cat_livre, ["R_TE", "R_TUSD_CAT", "KWH_TE", "KW_CAT"]] = 0.0
sap["ORIGEM"] = "SAP ZUBIL_146"
sap = pd.concat([sap, parte_cativa], ignore_index=True)

cativo = sap["MERCADO"] == "CATIVO"
sap["MWH"] = np.where(cativo, sap["KWH_TE"], sap["KWH_TUSD_LIV"]) / 1000.0
sap["KW"] = np.where(cativo & sap["SUBGRUPO"].str.startswith("A"), sap["KW_CAT"], 0.0)
sap["VALOR_RELATORIO"] = sap["R_TE"] + sap["R_TUSD_CAT"] + sap["R_TUSD_LIV"]
print(f"TE e TUSD TE em categorias livres, tratados como cativo (convencao do relatorio): "
      f"R$ {parte_cativa[['R_TE', 'R_TUSD_CAT']].sum().sum():,.2f}, "
      f"{parte_cativa['KWH_TE'].sum() / 1000:,.1f} MWh, {parte_cativa['KW_CAT'].sum():,.0f} kW")

nao_tarif = sap.groupby(["MERCADO", "CLASSE_REL"], as_index=False)[["R_REATIVO", "R_BANDEIRA"]].sum()
print(f"{len(z)} linhas do ZUBIL -> {len(sap)} linhas por classe, subclasse e categoria")
print(sap.groupby("MERCADO")[["MWH", "KW", "VALOR_RELATORIO"]].sum().to_string(float_format="{:,.1f}".format))
print(f"reativo R$ {sap['R_REATIVO'].sum():,.2f} | bandeira R$ {sap['R_BANDEIRA'].sum():,.2f}")

---
## Passo 4 — Ajustes manuais

A contabilidade completa a RNF do sistema com ajustes calculados fora do SAP. Para cada um, o
relatório traz o volume por classe (`RNF_POR_CLASSE`) e o valor total (`RESUMO`). O valor de cada
classe é estimado como volume × tarifa média do ZUBIL da classe — o mesmo critério da contabilidade
— e ajustado para somar o total do ajuste.

Dentro da classe, o ajuste herda o **mix do SAP**: é distribuído pelas linhas do ZUBIL da mesma classe
e mercado, na proporção da energia (volume e demanda) e do valor de cada linha. A iluminação pública
não existe no ZUBIL e recebe linha própria, na tarifa B4a.

A bandeira não faturada do ajuste vai para os não tarifáveis.

In [ ]:
CLASSES_REL = ["RESIDENCIAL", "INDUSTRIAL", "COMERCIAL", "RURAL", "PODER PÚBLICO",
               "ILUMINAÇÃO PÚBLICA", "SERVIÇO PÚBLICO"]            # columns C..I of RNF_POR_CLASSE
# (name, label in RNF_POR_CLASSE, label in RESUMO, market)
AJUSTES = [("TOI",              "RNF TOI SP",        "RNF TOI",                            "CATIVO"),
           ("Faturas pendentes", "RNF PENDENTES",    "RNF Pendentes (Faturas represadas)", "CATIVO"),
           ("Art. 323",          "RNF ARTIGO 323",   "RNF Faturas art.323",                "CATIVO"),
           ("Iluminação pública", "RNF - IP",        "RNF - IP - GRUPO B",                 "CATIVO"),
           ("Ajuste de carga",   "Ajuste carga",     "Ajuste Faturamento Cativo",          "CATIVO"),
           ("Ajuste livre",      "RNF Ajuste Livres", "Ajuste Carga livre",                "LIVRE")]
BANDEIRA_AJUSTE = num(acha_linha(resumo, "Bandeira RNF").get(2))

# average tariff of the SAP lines per market and class (R$/MWh)
tm = sap.groupby(["MERCADO", "CLASSE_REL"])[["VALOR_RELATORIO", "MWH"]].sum()
tm = (tm["VALOR_RELATORIO"] / tm["MWH"]).replace([np.inf, -np.inf], np.nan)

aj_classe = []
for nome, rot_cls, rot_res, merc in AJUSTES:
    l = acha_linha(por_classe, rot_cls)
    mwh = {c: num(l.get(2 + j)) / 1000.0 for j, c in enumerate(CLASSES_REL)}
    valor = num(acha_linha(resumo, rot_res).get(2))
    aj_classe.append({"AJUSTE": nome, "MERCADO": merc, "VALOR_TOTAL": valor, **mwh})
aj_classe = pd.DataFrame(aj_classe).set_index("AJUSTE")

# IP has no SAP lines: its average tariff is the one of the IP adjustment itself
ip = aj_classe.loc["Iluminação pública"]
tm.loc[("CATIVO", "ILUMINAÇÃO PÚBLICA")] = ip["VALOR_TOTAL"] / ip["ILUMINAÇÃO PÚBLICA"]

linhas_aj, sem_destino = [], []
for nome, r in aj_classe.iterrows():
    merc = r["MERCADO"]
    peso = pd.Series({c: r[c] * tm.get((merc, c), np.nan) for c in CLASSES_REL}).fillna(0.0)
    valor_cls = peso / peso.sum() * r["VALOR_TOTAL"] if peso.sum() else peso
    for c in CLASSES_REL:
        mwh_c, val_c = r[c], valor_cls[c]
        if not mwh_c and not val_c:
            continue
        if c == "ILUMINAÇÃO PÚBLICA":
            linhas_aj.append({"ORIGEM": nome, "MERCADO": merc, "CLASSE_REL": c, "CL1": "IP",
                              "SUB_SAP": "B4a", "CATEGORIA": "B4A (ajuste)", "SUBGRUPO": "B4",
                              "MODALIDADE": "Convencional", "BENEFICIO": None,
                              "MWH": mwh_c, "KW": 0.0, "VALOR_RELATORIO": val_c})
            continue
        ref = sap[(sap["MERCADO"] == merc) & (sap["CLASSE_REL"] == c) & (sap["MWH"] > 0)]
        if ref.empty:
            sem_destino.append((nome, c, mwh_c, val_c))
            continue
        f_e = ref["MWH"] / ref["MWH"].sum()
        f_v = ref["VALOR_RELATORIO"] / ref["VALOR_RELATORIO"].sum()
        novo = ref[["CL1", "SUB_SAP", "CATEGORIA", "SUBGRUPO", "MODALIDADE", "MERCADO",
                    "CLASSE_REL", "BENEFICIO"]].copy()
        novo["ORIGEM"] = nome
        novo["MWH"] = mwh_c * f_e
        novo["KW"] = ref["KW"] * (novo["MWH"] / ref["MWH"])     # demand follows the energy share
        novo["VALOR_RELATORIO"] = val_c * f_v
        linhas_aj.extend(novo.to_dict("records"))

ajustes = pd.DataFrame(linhas_aj)
print(aj_classe.assign(MWH=aj_classe[CLASSES_REL].sum(axis=1))[["MERCADO", "MWH", "VALOR_TOTAL"]]
      .to_string(float_format="{:,.1f}".format))
print(f"bandeira dos ajustes (nao tarifavel): R$ {BANDEIRA_AJUSTE:,.2f}")
if sem_destino:
    print("ATENCAO: ajuste sem linha do SAP na mesma classe e mercado (fica fora da abertura):")
    for s in sem_destino:
        print(f"   {s[0]} | {s[1]} | {s[2]:,.1f} MWh | R$ {s[3]:,.2f}")

---
## Passo 4b — Retirar PIS/COFINS

Os valores do ZUBIL — e, por consequência, os ajustes manuais, valorizados pela tarifa média do
ZUBIL — **contêm PIS/COFINS** (confirmado pelo usuário em 14/10/2026). A tarifa homologada não tem
tributos, então o alvo do bloco 2 é o valor **líquido**:

`valor líquido = valor do relatório × (1 − alíquota efetiva de PIS/COFINS do mês)`

A alíquota é a do próprio relatório: aba `RNF TOI SP`, células "Alíquota de COFINS" e "Alíquota de
PIS" (C34 e C35 em agosto/2026).
O tributo retirado vai para os não tarifáveis e volta na conciliação.

In [ ]:
if ALIQUOTA_PIS_COFINS is None:
    cofins = num(acha_linha(toi, "Alíquota de COFINS").get(2))
    pis = num(acha_linha(toi, "Alíquota de PIS").get(2))
    ALIQUOTA_PIS_COFINS = cofins + pis
    print(f"aliquotas do relatorio (aba RNF TOI SP): COFINS {cofins:.4%} + PIS {pis:.4%}")
print(f"aliquota efetiva de PIS/COFINS: {ALIQUOTA_PIS_COFINS:.4%}")

for df in (sap, ajustes):
    df["VALOR_BRUTO"] = df["VALOR_RELATORIO"]
    df["PIS_COFINS"] = df["VALOR_BRUTO"] * ALIQUOTA_PIS_COFINS
    df["VALOR_RELATORIO"] = df["VALOR_BRUTO"] - df["PIS_COFINS"]      # net target from here on
PIS_COFINS_RETIRADO = sap["PIS_COFINS"].sum() + ajustes["PIS_COFINS"].sum()
print(f"PIS/COFINS retirado do alvo: R$ {PIS_COFINS_RETIRADO:,.2f}")

---
## Passo 5 — Chave tarifária

| Origem | Vai para a chave como |
|---|---|
| Grupo A e B3 | classe e subclasse `Não se aplica` (a tarifa não varia por classe) |
| B1 | classe `Residencial`; subclasse `Residencial`, ou `Baixa Renda` / Desconto Social pelo benefício (Passo 8) |
| B2 / B4 | classe `Rural` / `Iluminação pública`; subclasse `Não se aplica` / `Iluminação pública – B4a` |
| Detalhe e acessante | `Não se aplica` (o ZUBIL não identifica APE, SCEE nem fonte incentivada: o efeito deles fica no bloco 2) |

In [ ]:
NSA = "Não se aplica"
linhas = pd.concat([sap.drop(columns=[c for c in VALORES if c in sap.columns]), ajustes],
                   ignore_index=True)
linhas["CLASSE"] = linhas["SUBGRUPO"].map({"B1": "Residencial", "B2": "Rural",
                                           "B4": "Iluminação pública"}).fillna(NSA)
linhas["SUBCLASSE"] = np.select(
    [linhas["SUBGRUPO"] == "B1", linhas["SUBGRUPO"] == "B4"],
    ["Residencial", "Iluminação pública – B4a"], NSA)
linhas["UNIDADE_E"], linhas["DETALHE"], linhas["ACESSANTE"] = "MWh", NSA, NSA
print(linhas.groupby(["ORIGEM", "MERCADO"])[["MWH", "KW", "VALOR_RELATORIO"]].sum()
      .to_string(float_format="{:,.1f}".format))

---
## Passo 6 — Posto horário, pelo Faturamento do mesmo mês

O ZUBIL não separa ponta e fora ponta. A divisão vem do P × Q do Faturamento do mesmo mês
(`pxq_<ref>.xlsx`): a participação de cada posto no consumo (e, no Azul, na demanda) faturado, no
grupo mais próximo com volume:

1. subgrupo, modalidade, mercado e classe;
2. subgrupo, modalidade e mercado;
3. subgrupo e modalidade.

Vale para energia do grupo A (Azul e Verde) e da Branca (com o intermediário), e para a demanda do
Azul. Convencional e demanda do Verde não têm posto.

In [ ]:
def sem_acento(t):
    n = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in n if not unicodedata.combining(c)).lower()

def classe_bw(t):
    s = sem_acento(t)
    for chave, cl in (("resid", "RESIDENCIAL"), ("comerc", "COMERCIAL"), ("indust", "INDUSTRIAL"),
                      ("rural", "RURAL"), ("poder", "PODER PÚBLICO"), ("ilumin", "ILUMINAÇÃO PÚBLICA"),
                      ("servi", "SERVIÇO PÚBLICO")):
        if chave in s:
            return cl
    return None

POSTO_COD = {"EP": "Ponta", "EFP": "Fora ponta", "EINT": "Intermediário",
             "DP": "Ponta", "DFP": "Fora ponta"}
bw = pd.read_excel(ARQ_PXQ_BW, sheet_name="Componentes",
                   usecols=["Chave", "Classe principal", "Flag Cli.Liv./ Cat.", "Tarifável?",
                            "Consumo TUSD (kWh)", "Consumo TE (kWh)", "Demanda (kW)"])
bw = bw[bw["Tarifável?"] == "SIM"].copy()
bw["MERCADO"] = bw["Flag Cli.Liv./ Cat."].map({"Cliente Cativo": "CATIVO", "Cliente Livre": "LIVRE",
                                               "Cliente Híbrido": "LIVRE"})
cab_ch = bw["Chave"].astype(str).str.extract(r"^(A2|A3a|A4|AS|B1|B2|B3|B4)(Azul|Verde|Convencional|Branca)")
bw["SUBGRUPO"], bw["MODALIDADE"] = cab_ch[0], cab_ch[1]
bw["POSTO"] = bw["Chave"].astype(str).str.extract(r"(EINT|EFP|EP|DFP|DP)(?:MWh|kW)$")[0].map(POSTO_COD)
bw["CLASSE_REL"] = bw["Classe principal"].map(classe_bw)
bw["E"] = bw["Consumo TUSD (kWh)"].fillna(0) + bw["Consumo TE (kWh)"].fillna(0)
bw["D"] = bw["Demanda (kW)"].fillna(0)
bw = bw[bw["MERCADO"].notna() & bw["POSTO"].notna()]

NIVEIS = [["SUBGRUPO", "MODALIDADE", "MERCADO", "CLASSE_REL"],
          ["SUBGRUPO", "MODALIDADE", "MERCADO"], ["SUBGRUPO", "MODALIDADE"]]

def partes_posto(r, grandeza):
    """{posto: share} for a line, and the level used; grandeza 'E' (energy) or 'D' (demand)."""
    for n, chaves in enumerate(NIVEIS, 1):
        ref = bw
        for k in chaves:
            ref = ref[ref[k] == r[k]]
        tot = ref[grandeza].sum()
        if tot > 0:
            return (ref.groupby("POSTO")[grandeza].sum() / tot).to_dict(), n
    return None, None

def precisa_posto(r, grandeza):
    if grandeza == "E":
        return r["MODALIDADE"] == "Branca" or (r["SUBGRUPO"][0] == "A" and r["MODALIDADE"] in ("Azul", "Verde"))
    return r["MODALIDADE"] == "Azul"

linhas["POSTOS_E"], linhas["POSTOS_D"], linhas["NIVEL_POSTO"] = None, None, ""
for i, r in linhas.iterrows():
    niv = []
    for g, col, padrao in (("E", "POSTOS_E", {NSA: 1.0}), ("D", "POSTOS_D", {NSA: 1.0})):
        if g == "D" and r["KW"] == 0:
            continue
        if precisa_posto(r, g):
            p, n = partes_posto(r, g)
            if p is None:
                raise ValueError(f"sem referencia de posto no BW para {r['SUBGRUPO']} {r['MODALIDADE']}")
            linhas.at[i, col] = p
            niv.append(f"{g}{n}")
        else:
            linhas.at[i, col] = padrao
    linhas.at[i, "NIVEL_POSTO"] = " ".join(niv) if niv else "sem posto"
linhas["PCT_PONTA_E"] = linhas["POSTOS_E"].map(lambda p: p.get("Ponta", 0.0))
print(linhas.groupby(["SUBGRUPO", "MODALIDADE", "MERCADO"])["PCT_PONTA_E"].mean()
      .map("{:.1%}".format).to_string())

---
## Passo 7 — Tarifa da vigência

Valem as vigências em vigor no mês de referência. No mês comum há uma só. No mês do reajuste (4 de
julho, em SP) há duas, e o peso de cada uma por subgrupo é estimado no Passo 8, pela mesma regra da
tarifa proporcional do Faturamento (README do BW, 3.6; REN 1.000/2021, art. 173, § 2º). As quatro
camadas vêm da mesma tabela (TA = BE + CVA + FIN em cada item).

In [ ]:
CAMADAS = ["TA", "BE", "CVA", "FIN"]
ATRIBUTOS = ["SUBGRUPO", "MODALIDADE", "CLASSE", "SUBCLASSE", "DETALHE", "ACESSANTE", "POSTO", "UNIDADE"]

tl = pd.read_csv(ARQ_TARIFA_LONG, sep=";", decimal=",", encoding="utf-8-sig")
tl["VIGENCIA_INICIO"] = pd.to_datetime(tl["VIGENCIA_INICIO"])
tl["VIGENCIA_FIM"] = pd.to_datetime(tl["VIGENCIA_FIM"])
COMPONENTES = [c for c in tl.columns if c not in {"ANO_VIGENCIA", "VIGENCIA_INICIO", "VIGENCIA_FIM",
               "CAMADA", "ORIGEM", "Total TUSD", "Total TE", *ATRIBUTOS}]

# vintages in force during the month, with the calendar weight (days in the month)
ini, fim = ALVO, ALVO + pd.offsets.MonthEnd(0)
dias_mes = (fim - ini).days + 1
VIGENCIAS = {}
for ano, v in tl.groupby("ANO_VIGENCIA")[["VIGENCIA_INICIO", "VIGENCIA_FIM"]].first().iterrows():
    dias = (min(v["VIGENCIA_FIM"], fim) - max(v["VIGENCIA_INICIO"], ini)).days + 1
    if dias > 0:
        VIGENCIAS[int(ano)] = dias / dias_mes
assert VIGENCIAS, f"nenhuma vigencia cobre {REFERENCIA}"
TARIFAS = {}
for ano in VIGENCIAS:
    _t = tl[tl["ANO_VIGENCIA"] == ano]
    TARIFAS[ano] = {c: _t[_t["CAMADA"] == c].set_index(ATRIBUTOS)[COMPONENTES].astype(float) for c in CAMADAS}
ANO_VIGENCIA = " + ".join(str(a) for a in VIGENCIAS)
print("vigencias no mes (peso de calendario): " +
      ", ".join(f"{a}: {p:.1%}" for a, p in VIGENCIAS.items()))

def tarifa(attrs, postos, unidade, subclasse=None, pesos=None):
    """Tariff per layer weighted by posto shares and vintage weights; falls back to subclass
    'Nao se aplica'; a key missing in one vintage takes the other vintages' weight."""
    pesos = pesos or VIGENCIAS
    saida = {c: np.zeros(len(COMPONENTES)) for c in CAMADAS}
    for posto, peso in postos.items():
        base = {**attrs, "POSTO": posto, "UNIDADE": unidade}
        if subclasse:
            base["SUBCLASSE"] = subclasse
        achados = []
        for ano, p in pesos.items():
            if p <= 0:
                continue
            idx = TARIFAS[ano]["TA"].index
            k = tuple(base[a] for a in ATRIBUTOS)
            if k not in idx:
                k = tuple({**base, "SUBCLASSE": NSA}[a] for a in ATRIBUTOS)
            if k in idx:
                achados.append((ano, p, k))
        if not achados:
            raise KeyError(f"sem tarifa: {tuple(base[a] for a in ATRIBUTOS)}")
        tot = sum(p for _, p, _ in achados)
        for ano, p, k in achados:
            for c in CAMADAS:
                saida[c] = saida[c] + peso * (p / tot) * TARIFAS[ano][c].loc[k].values
    return saida

---
## Passo 8 — Bloco 1: o valor pela tarifa

- **Tarifa proporcional (mês do reajuste):** o peso da vigência nova em cada subgrupo é o que
  reproduz o valor líquido das linhas cativas do SAP sem benefício social:
  `w = (R − A) / (B − A)`, limitado a [0, 1], com A e B o bloco 1 na vigência antiga e na nova. Se as
  duas tarifas diferem menos de 1%, vale o peso de calendário. Subgrupos sem linha de calibração
  (B4, por exemplo) também usam o calendário.
- **Parcelas:** cativo paga TUSD e TE na energia; livre, só TUSD. A demanda é só TUSD.
- **Benefícios sociais:** o ZUBIL não traz a faixa de consumo. Como no Faturamento (procedimento do
  BW, Passo 8), cada linha de benefício é valorizada nas duas faixas e a participação da faixa 02 é a
  que reproduz o valor do relatório: `s2 = (R − A) / (B − A)`, limitada a [0, 1].
  - Tarifa Social: faixa 01 = gratuidade (A = 0); faixa 02 = tarifa `Baixa Renda`.
  - Desconto Social: faixa 01 e faixa 02 pelas chaves próprias.

In [ ]:
ATTR_FIXOS = ["SUBGRUPO", "MODALIDADE", "CLASSE", "SUBCLASSE", "DETALHE", "ACESSANTE"]
COLS_TE = np.array([c.startswith("TE") for c in COMPONENTES])

def valor_linha(r, subclasse=None, pesos=None):
    """Block-1 value per layer for one line (energy + demand), optional subclass and vintage weights."""
    pesos = pesos or PESO_VIG.get(r["SUBGRUPO"], VIGENCIAS)
    attrs = {a: r[a] for a in ATTR_FIXOS}
    te_ok = r["MERCADO"] == "CATIVO"
    e = tarifa(attrs, r["POSTOS_E"], "MWh", subclasse, pesos)
    out = {}
    for c in CAMADAS:
        v = e[c] * r["MWH"]
        if not te_ok:
            v = np.where(COLS_TE, 0.0, v)
        out[c] = v
    if r["KW"]:
        d = tarifa({**attrs, "CLASSE": NSA, "SUBCLASSE": NSA}, r["POSTOS_D"], "kW", None, pesos)
        for c in CAMADAS:
            out[c] = out[c] + np.where(COLS_TE, 0.0, d[c] * r["KW"])
    return out

# Vintage weight per subgroup in the readjustment month (same rule as the Faturamento, Passo 7b):
# the share of the new tariff that reproduces the net value of the captive SAP lines without social
# benefit, w = (R - A) / (B - A), limited to [0, 1]; calendar weight when the tariffs barely differ.
PESO_VIG, CALIBRACAO = {}, None
LIMIAR_DIF_TARIFA = 0.01
if len(VIGENCIAS) == 2:
    antiga, nova = sorted(VIGENCIAS)
    base_cal = linhas[(linhas["ORIGEM"] == "SAP ZUBIL_146") & (linhas["MERCADO"] == "CATIVO")
                      & linhas["BENEFICIO"].isna()]
    cal = []
    for sg, g in base_cal.groupby("SUBGRUPO"):
        A = sum(valor_linha(r, pesos={antiga: 1.0})["TA"].sum() for _, r in g.iterrows())
        B = sum(valor_linha(r, pesos={nova: 1.0})["TA"].sum() for _, r in g.iterrows())
        R = g["VALOR_RELATORIO"].sum()
        if A and abs(B - A) / A >= LIMIAR_DIF_TARIFA:
            w, regra = float(np.clip((R - A) / (B - A), 0.0, 1.0)), "estimado pelo valor"
        else:
            w, regra = VIGENCIAS[nova], "calendario (tarifas quase iguais)"
        PESO_VIG[sg] = {antiga: 1.0 - w, nova: w}
        cal.append({"SUBGRUPO": sg, "VALOR_LIQUIDO": R, "BLOCO1_ANTIGA": A, "BLOCO1_NOVA": B,
                    f"PESO_{nova}": w, "REGRA": regra})
    CALIBRACAO = pd.DataFrame(cal)
    print(f"tarifa proporcional: peso de calendario da vigencia {nova} = {VIGENCIAS[nova]:.1%}")
    print(CALIBRACAO.to_string(index=False, float_format="{:,.3f}".format))

B1 = {c: np.zeros((len(linhas), len(COMPONENTES))) for c in CAMADAS}
linhas["FAIXA2_PCT"] = np.nan
for i, r in linhas.iterrows():
    ben = r["BENEFICIO"]
    if ben == "BAIXA_RENDA":
        A = {c: np.zeros(len(COMPONENTES)) for c in CAMADAS}
        B = valor_linha(r, "Baixa Renda")
    elif ben == "DESCONTO_SOCIAL":
        A = valor_linha(r, "Residencial Desconto Social – faixa 01")
        B = valor_linha(r, "Residencial Desconto Social – faixa 02")
    else:
        v = valor_linha(r)
        for c in CAMADAS:
            B1[c][i] = v[c]
        continue
    a, b, R = A["TA"].sum(), B["TA"].sum(), r["VALOR_RELATORIO"]
    s2 = float(np.clip((R - a) / (b - a), 0.0, 1.0)) if b != a else 1.0
    linhas.at[i, "FAIXA2_PCT"] = s2
    for c in CAMADAS:
        B1[c][i] = (1 - s2) * A[c] + s2 * B[c]

B1 = {c: pd.DataFrame(B1[c], index=linhas.index, columns=COMPONENTES) for c in CAMADAS}
linhas["BLOCO1_TA"] = B1["TA"].sum(axis=1)
linhas["DIFERENCA"] = linhas["VALOR_RELATORIO"] - linhas["BLOCO1_TA"]
g = linhas.groupby("MERCADO")[["VALOR_RELATORIO", "BLOCO1_TA", "DIFERENCA"]].sum()
g["DIF_%"] = g["DIFERENCA"] / g["VALOR_RELATORIO"]
print(g.to_string(float_format="{:,.2f}".format))

---
## Passo 9 — Bloco 2: levar ao valor do relatório

A mesma regra do Faturamento (README do BW, seção 4):

- **Livre do grupo A — regime fio.** A diferença vai para Fio A e Fio B, na composição da tarifa de
  demanda do mesmo subgrupo e modalidade. Ela vem da demanda (que o ZUBIL do livre não permite
  valorizar) e dos descontos de fonte incentivada e APE, que incidem sobre o fio.
- **Demais linhas — rateio proporcional.** A diferença é distribuída por todos os itens e camadas na
  proporção do bloco 1, como na 2ª passada do Faturamento.

Linha com bloco 1 zero e valor no relatório não tem o que escalar: fica como resíduo, informado à
parte.

In [ ]:
FAMILIA = {
    "FIO A":   ["TUSD_RB", "TUSD_FR", "TUSD_CCT", "TUSD_CCD", "TUSD_CUSD", "TUSDG_T",
                "TE_TRANSPORTE_ITAIPU", "TE_TUST_ITAIPU", "TE_TUST_CI"],
    "FIO B":   ["TUSD_FioB"],
    "ENCARGOS":["TUSD_CDE_COVID", "TUSD_TFSEE", "TUSD_PeD", "TUSD_ONS", "TUSD_CCC", "TUSD_CDE",
                "TUSD_PROINFA", "Liminar1", "TUSDG_ONS", "TE_CDE_COVID", "TE_CDE_ELET",
                "TE_CDE_GD", "TE_PeD", "TE_ESSERR", "TE_CFURH"],
    "PERDAS":  ["TUSD_PT", "TUSD_Per_RB_D", "TUSD_PNT", "TE_Per_RB"],
    "ENERGIA": ["TE_ENERGIA", "TE_ANGRA"],
    "RI":      ["TUSD_RI"],
    "OUTROS":  ["TUSD Subsidio", "TUSD BENEFICIO_L14299", "TUSD Outros",
                "TE Subsidio", "TE BENEFICIO_L14299"],
}
assert not [c for c in COMPONENTES if c not in {x for v in FAMILIA.values() for x in v}]
FIO = np.array([c in FAMILIA["FIO A"] + FAMILIA["FIO B"] for c in COMPONENTES])

B2 = {c: B1[c].copy() for c in CAMADAS}
linhas["REGIME"] = np.where((linhas["MERCADO"] == "LIVRE") & linhas["SUBGRUPO"].str.startswith("A"),
                            "fio", "proporcional")
linhas["AJUSTE_FIO"], linhas["FATOR"], linhas["RESIDUO"] = 0.0, 1.0, 0.0
for i, r in linhas.iterrows():
    dif = r["DIFERENCA"]
    if r["REGIME"] == "fio":
        p = partes_posto(r, "D")[0] if r["MODALIDADE"] == "Azul" else {NSA: 1.0}
        d = tarifa({"SUBGRUPO": r["SUBGRUPO"], "MODALIDADE": r["MODALIDADE"], "CLASSE": NSA,
                    "SUBCLASSE": NSA, "DETALHE": NSA, "ACESSANTE": NSA}, p, "kW", None,
                   PESO_VIG.get(r["SUBGRUPO"]))
        base = (d["TA"] * FIO).sum()
        for c in CAMADAS:
            B2[c].loc[i] = B1[c].loc[i].values + dif * (d[c] * FIO) / base
        linhas.at[i, "AJUSTE_FIO"] = dif
    elif r["BLOCO1_TA"]:
        f = r["VALOR_RELATORIO"] / r["BLOCO1_TA"]
        for c in CAMADAS:
            B2[c].loc[i] = B1[c].loc[i] * f
        linhas.at[i, "FATOR"] = f
    else:
        linhas.at[i, "RESIDUO"] = r["VALOR_RELATORIO"]
linhas["BLOCO2_TA"] = B2["TA"].sum(axis=1)
print(linhas.groupby("REGIME")[["VALOR_RELATORIO", "BLOCO1_TA", "AJUSTE_FIO", "BLOCO2_TA", "RESIDUO"]]
      .sum().to_string(float_format="{:,.2f}".format))

---
## Passo 10 — Conferências

1. Bloco 2 + resíduo = valor do relatório, linha a linha.
2. TA = BE + CVA + FIN no bloco 2.
3. Famílias = componentes.
4. Volume aberto x volume da aba `RNF_POR_CLASSE` — informativa: em alguns meses o pivô dessa aba não
   foi atualizado no relatório (sessions, P12). O controle de valor é a conciliação do Passo 11.
5. Medida, não aprovação: fator do rateio proporcional por subgrupo (quanto a tarifa explica sozinha),
   separando os valores cativos lançados em categoria livre.

In [ ]:
ok = True
d = (linhas["BLOCO2_TA"] + linhas["RESIDUO"] - linhas["VALOR_RELATORIO"]).abs().max()
print(f"1. bloco 2 + residuo x relatorio ......... diferenca maxima {d:,.6f}"); ok &= d < 0.01
d = (B2["TA"] - B2["BE"] - B2["CVA"] - B2["FIN"]).abs().max().max()
print(f"2. TA = BE + CVA + FIN ................... diferenca maxima {d:,.8f}"); ok &= d < 0.01
d = max(abs(B2[c].sum().sum() - sum(B2[c][l].sum().sum() for l in FAMILIA.values())) for c in CAMADAS)
print(f"3. familias x componentes ................ diferenca maxima {d:,.8f}"); ok &= d < 0.01
# informative: in some months the report's own class pivot is not refreshed (sessions, P12)
for merc, rot in (("CATIVO", "RNF Cativo"), ("LIVRE", "RNF Livre")):
    rel = num(acha_linha(por_classe, rot).get(11))
    d = linhas.loc[linhas["MERCADO"] == merc, "MWH"].sum() - rel
    aviso = "" if abs(d) < 1.0 else "  (aviso: pivo RNF_POR_CLASSE do relatorio difere do ZUBIL)"
    print(f"4. volume {merc:<6} x RNF_POR_CLASSE ..... diferenca {d:,.3f} MWh{aviso}")
print(f"   residuo nao ajustavel ................ R$ {linhas['RESIDUO'].sum():,.2f}")
em_cat_livre = linhas["CATEGORIA"].map(lambda c: CATEGORIA.get(c, ("", "", ""))[2]) == "LIVRE"
prop = (linhas[(linhas["REGIME"] == "proporcional") & ~em_cat_livre]
        .groupby(["SUBGRUPO", "MODALIDADE"])[["VALOR_RELATORIO", "BLOCO1_TA"]].sum())
print("\n5. relatorio / bloco 1 no rateio proporcional, categorias cativas:")
print((prop["VALOR_RELATORIO"] / prop["BLOCO1_TA"]).map("{:.3f}".format).to_string())
ccl = linhas[(linhas["MERCADO"] == "CATIVO") & em_cat_livre][["VALOR_RELATORIO", "BLOCO1_TA"]].sum()
print(f"   valores cativos em categoria livre: relatorio / bloco 1 = {ccl['VALOR_RELATORIO'] / ccl['BLOCO1_TA']:.3f}")
print("\nTODAS AS CONFERENCIAS PASSARAM" if ok else "\nALGUMA CONFERENCIA FALHOU - NAO USAR")

---
## Passo 11 — Conciliação com o relatório

O total aberto mais os não tarifáveis (PIS/COFINS retirado no Passo 4b, reativo e bandeira) reproduz
o **valor contabilizado** na conta 1120110910 (aba `Conciliação contábil Novo`, linha
"Cons-Forn.n/faturado", coluna I). O total do `Comparativo Novo` é mostrado só como informação: em
alguns meses ele difere do contabilizado, e o próprio relatório acusa a diferença no "Check".

In [ ]:
# booked value: account 1120110910, column "VALOR APOS AJUSTE" of "Conciliacao contabil Novo"
lin_cont = acha_linha(contabil, "Cons-Forn.n/faturado", col=3)
CONTABILIZADO = lin_cont.get(8) if isinstance(lin_cont.get(8), float) else None
abertura = pd.Series({"Valor aberto por item (liquido)": linhas["BLOCO2_TA"].sum() + linhas["RESIDUO"].sum(),
                      "PIS/COFINS retirado": PIS_COFINS_RETIRADO, "Reativo": sap["R_REATIVO"].sum(),
                      "Bandeira": sap["R_BANDEIRA"].sum() + BANDEIRA_AJUSTE})
abertura["Total"] = abertura.sum()
nf = acha_linha(comparativo, "Não Faturado", col=0)
CONCILIACAO = pd.DataFrame({"R$": abertura})
CONCILIACAO.loc["Contabilizado (Conciliação contábil Novo, coluna I)"] = CONTABILIZADO if CONTABILIZADO else np.nan
CONCILIACAO.loc["Diferença x contabilizado"] = abertura["Total"] - CONTABILIZADO if CONTABILIZADO else np.nan
CONCILIACAO.loc["Comparativo Novo, Não Faturado (informativo)"] = nf[8]
CONCILIACAO.loc["Diferença x Comparativo Novo"] = abertura["Total"] - nf[8]
print(CONCILIACAO["R$"].map("{:,.2f}".format).to_string())
if CONTABILIZADO is None:
    print("ATENCAO: valor contabilizado ilegivel no relatorio (erro de formula); conciliado com o Comparativo Novo")
    assert abs(abertura["Total"] - nf[8]) < 1.0, "conciliacao com o relatorio nao fecha"
else:
    assert abs(abertura["Total"] - CONTABILIZADO) < 1.0, "conciliacao com o valor contabilizado nao fecha"
    if abs(abertura["Total"] - nf[8]) >= 1.0:
        print("ATENCAO: o Comparativo Novo difere do valor contabilizado (ver 'Check' do proprio relatorio)")

---
## Passo 12 — Gravar a saída

`1-Entrega/rnf/rnf_<REFERENCIA>.xlsx`, no desenho do `pxq_<ref>.xlsx`:

| Aba | Conteúdo |
|---|---|
| `Componentes` | uma linha por origem, classe, subclasse e categoria: bloco 1 e bloco 2 na tarifa de aplicação, 37 itens |
| `Componentes_BE` | bloco 2 na base econômica |
| `Familias` | bloco 2 nas sete famílias e quatro camadas |
| `Resumo` | MWh, valor e tarifa média por origem, mercado e classe |
| `Conciliacao` | fechamento com o valor contabilizado (e comparação com o `Comparativo Novo`) |
| `Nao_tarifaveis` | PIS/COFINS retirado, reativo e bandeira por mercado e classe |
| `Calibracao_vigencia` | só no mês do reajuste: peso de cada vigência por subgrupo |

In [ ]:
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

IDENT = ["ORIGEM", "MERCADO", "CLASSE_REL", "CL1", "SUB_SAP", "CATEGORIA", "SUBGRUPO", "MODALIDADE",
         "SUBCLASSE", "BENEFICIO", "NIVEL_POSTO", "PCT_PONTA_E", "MWH", "KW", "VALOR_BRUTO", "PIS_COFINS", "VALOR_RELATORIO",
         "BLOCO1_TA", "DIFERENCA", "REGIME", "AJUSTE_FIO", "FATOR", "FAIXA2_PCT", "RESIDUO", "BLOCO2_TA"]
ROTULO = {"ORIGEM": "Origem", "MERCADO": "Mercado", "CLASSE_REL": "Classe", "CL1": "Classe SAP",
          "SUB_SAP": "Subclasse SAP", "CATEGORIA": "Categoria tarifa", "SUBGRUPO": "Subgrupo",
          "MODALIDADE": "Modalidade", "SUBCLASSE": "Subclasse (tarifa)", "BENEFICIO": "Benefício social",
          "NIVEL_POSTO": "Nível da divisão por posto", "PCT_PONTA_E": "% energia na ponta",
          "MWH": "RNF (MWh)", "KW": "Demanda (kW)", "VALOR_BRUTO": "Valor do relatório, com PIS/COFINS (R$)",
          "PIS_COFINS": "PIS/COFINS (R$)", "VALOR_RELATORIO": "Valor líquido, alvo do bloco 2 (R$)",
          "BLOCO1_TA": "Bloco 1 TA (R$)", "DIFERENCA": "Diferença (R$)", "REGIME": "Regime do bloco 2",
          "AJUSTE_FIO": "Ajuste no fio (R$)", "FATOR": "Fator proporcional",
          "FAIXA2_PCT": "% energia na faixa 02 (estimada)", "RESIDUO": "Resíduo (R$)",
          "BLOCO2_TA": "Bloco 2 TA (R$)"}
esq = linhas[IDENT].rename(columns=ROTULO)
esq.insert(0, "Referência", REFERENCIA)
esq.insert(1, "Vigência tarifária", ANO_VIGENCIA)

aba_comp = pd.concat([esq, B1["TA"].add_prefix("[1] "), B2["TA"].add_prefix("[2] ")], axis=1)
aba_be = pd.concat([esq, B2["BE"].add_prefix("[2] BE · ")], axis=1)
fam = pd.DataFrame({f"[2] {c} · {f}": B2[c][l].sum(axis=1) for c in CAMADAS for f, l in FAMILIA.items()})
aba_fam = pd.concat([esq, fam], axis=1)
aba_res = (linhas.groupby(["ORIGEM", "MERCADO", "CLASSE_REL"])[["MWH", "VALOR_BRUTO", "VALOR_RELATORIO", "BLOCO1_TA"]]
           .sum().reset_index())
aba_res["R$/MWh"] = aba_res["VALOR_RELATORIO"] / aba_res["MWH"]
aba_res = aba_res.rename(columns=ROTULO)
aba_nt = nao_tarif.rename(columns={"MERCADO": "Mercado", "CLASSE_REL": "Classe",
                                   "R_REATIVO": "Reativo (R$)", "R_BANDEIRA": "Bandeira (R$)"})
aba_nt.loc[len(aba_nt)] = ["CATIVO", "Ajustes manuais", 0.0, BANDEIRA_AJUSTE]
pc = linhas.groupby(["MERCADO", "CLASSE_REL"], as_index=False)["PIS_COFINS"].sum()
aba_nt = aba_nt.merge(pc.rename(columns={"MERCADO": "Mercado", "CLASSE_REL": "Classe",
                                         "PIS_COFINS": "PIS/COFINS retirado (R$)"}),
                      on=["Mercado", "Classe"], how="outer").fillna(0.0)

def formata(ws, n_ident):
    ws.freeze_panes = ws.cell(row=2, column=n_ident + 1)
    for cel in ws[1]:
        cel.font = Font(bold=True, size=9, color="FFFFFF")
        cel.fill = PatternFill("solid", fgColor="1F4E79")
        cel.alignment = Alignment(horizontal="center", wrap_text=True, vertical="center")
    ws.row_dimensions[1].height = 42
    for col in range(1, ws.max_column + 1):
        titulo = str(ws.cell(row=1, column=col).value)
        ws.column_dimensions[get_column_letter(col)].width = 18 if col <= n_ident else 15
        fmt = "0.0%" if "%" in titulo else ("0.0000" if "Fator" in titulo else "#,##0.00")
        for lin in range(2, ws.max_row + 1):
            if isinstance(ws.cell(row=lin, column=col).value, (int, float)):
                ws.cell(row=lin, column=col).number_format = fmt
    ws.auto_filter.ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"

DIR_SAIDA.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(SAIDA, engine="openpyxl") as w:
    n = esq.shape[1]
    for nome, df, ni in (("Componentes", aba_comp, n), ("Componentes_BE", aba_be, n),
                         ("Familias", aba_fam, n), ("Resumo", aba_res, 3),
                         ("Nao_tarifaveis", aba_nt, 2)):
        df.to_excel(w, sheet_name=nome, index=False)
        formata(w.book[nome], ni)
    CONCILIACAO.reset_index(names="Item").to_excel(w, sheet_name="Conciliacao", index=False)
    if CALIBRACAO is not None:
        CALIBRACAO.to_excel(w, sheet_name="Calibracao_vigencia", index=False)
    formata(w.book["Conciliacao"], 1)
print(f"gravado: {SAIDA}")